# 🏛️ ETL PIPELINE: SILVER TO GOLD DIMENSIONS
### Xây dựng các bảng Dimension giữ nguyên 100% tên cột và khóa tự nhiên gốc từ Silver data
---
**Nguyên tắc thiết kế:**
1. **Chỉ dùng Pandas & NumPy** để xử lý dữ liệu.
2. **Giữ nguyên 100% tên cột gốc** từ các file CSV trong thư mục `Silver data` (không đổi tên, không tạo thêm surrogate key nếu đã có khóa tự nhiên).
3. **Khóa chính (Primary Key) lấy trực tiếp từ thuộc tính có sẵn**:
   - `Dim_Customer` $
ightarrow$ Khóa chính: `customer_id`
   - `Dim_Product` $
ightarrow$ Khóa chính: `product_id`
   - `Dim_Geography` $
ightarrow$ Khóa chính: `zip`
   - `Dim_Promotion` $
ightarrow$ Khóa chính: `promo_id`
   - `Dim_Date` $
ightarrow$ Khóa chính: `order_date`

In [18]:
# ==============================================================================
# 1. KHỞI TẠO THƯ VIỆN PANDAS & NUMPY
# ==============================================================================
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print("✅ Đã khởi tạo thư viện Pandas & NumPy.")

✅ Đã khởi tạo thư viện Pandas & NumPy.


In [19]:
# ==============================================================================
# 2. ĐỌC DỮ LIỆU TỪ THƯ MỤC SILVER DATA VÀ LÀM SẠCH KÝ TỰ ẨN
# ==============================================================================
# Hàm hỗ trợ đọc file CSV và loại bỏ BOM
def load_csv(file_name):
    # Thử đọc từ thư mục hiện tại hoặc thư mục 'Silver data'
    for path in [file_name, f'Silver data/{file_name}', f'/content/{file_name}', f'/content/Silver data/{file_name}']:
        try:
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.replace('^\ufeff', '', regex=True).str.strip()
            print(f"✅ Đã nạp thành công: {path} | Shape: {df.shape}")
            return df
        except Exception:
            continue
    print(f"⚠️ Không tìm thấy file: {file_name}")
    return pd.DataFrame()

customers_df = load_csv('customers_silver.csv')
orders_df = load_csv('orders.csv')
products_df = load_csv('silver_product.csv')
geography_df = load_csv('geography.csv')
promotions_df = load_csv('promotions.csv')

✅ Đã nạp thành công: customers_silver.csv | Shape: (121930, 6)
✅ Đã nạp thành công: orders.csv | Shape: (646945, 9)
✅ Đã nạp thành công: silver_product.csv | Shape: (2412, 8)
✅ Đã nạp thành công: geography.csv | Shape: (39948, 4)
✅ Đã nạp thành công: promotions.csv | Shape: (50, 10)


---
## 3. TẠO CÁC BẢNG DIMENSION (GIỮ ĐÚNG TÊN CỘT GỐC)

In [20]:
# ==============================================================================
# 3.1. TẠO DIM_CUSTOMER (Khóa chính: customer_id)
# Cột gốc: customer_id, zip, signup_date, gender, age_group, acquisition_channel
# ==============================================================================
print("--- Đang tạo Dim_Customer ---")

dim_customer = customers_df[[
    'customer_id', 'zip', 'signup_date', 'gender', 'age_group', 'acquisition_channel'
]].drop_duplicates(subset=['customer_id']).copy()

# Chuẩn hóa dữ liệu giữ nguyên tên cột
dim_customer['signup_date'] = pd.to_datetime(dim_customer['signup_date'], errors='coerce')
dim_customer['gender'] = dim_customer['gender'].fillna('Unknown').astype(str).str.capitalize()
dim_customer['age_group'] = dim_customer['age_group'].fillna('Unknown').astype(str)
dim_customer['acquisition_channel'] = dim_customer['acquisition_channel'].fillna('direct').astype(str)
dim_customer['zip'] = dim_customer['zip'].astype(str).str.strip()

dim_customer = dim_customer.sort_values('customer_id').reset_index(drop=True)
print(f"✅ Dim_Customer: {dim_customer.shape[0]:,} khách hàng | Khóa chính: customer_id")
display(dim_customer.head())

--- Đang tạo Dim_Customer ---
✅ Dim_Customer: 121,930 khách hàng | Khóa chính: customer_id


,customer_id,zip,signup_date,gender,age_group,acquisition_channel
0,1,15201,2021-12-30,Female,35-44,social_media
1,2,15201,2013-12-27,Female,45-54,email_campaign
2,3,15201,2018-07-24,Female,18-24,organic_search
3,4,15201,2017-11-29,Male,35-44,referral
4,5,15201,2022-09-23,Male,55+,organic_search


In [21]:
# ==============================================================================
# 3.2. TẠO DIM_PRODUCT (Khóa chính: product_id)
# Cột gốc: product_id, product_name, category, segment, size, color, price, cogs
# ==============================================================================
print("--- Đang tạo Dim_Product ---")

dim_product = products_df[[
    'product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs'
]].drop_duplicates(subset=['product_id']).sort_values('product_id').reset_index(drop=True)

print(f"✅ Dim_Product: {dim_product.shape[0]:,} sản phẩm | Khóa chính: product_id")
display(dim_product.head())

--- Đang tạo Dim_Product ---
✅ Dim_Product: 2,412 sản phẩm | Khóa chính: product_id


,product_id,product_name,category,segment,size,color,price,cogs
0,1,DragonWear MA-01,Casual,All-weather,M,black,4945.500000,2732.883300
1,2,DragonWear MA-02,Casual,All-weather,L,orange,39.062877,22.660233
2,3,DragonWear MA-03,Casual,All-weather,XL,blue,10831.377190,10289.808330
3,4,DragonWear MA-04,Casual,All-weather,S,white,9610.756522,5604.032128
4,5,DragonWear MA-05,Casual,All-weather,M,purple,8946.000000,8498.700000


In [22]:
# ==============================================================================
# 3.3. TẠO DIM_GEOGRAPHY (Khóa chính: zip)
# Cột gốc: zip, city, region, district
# ==============================================================================
print("--- Đang tạo Dim_Geography ---")

geography_df['zip'] = geography_df['zip'].astype(str).str.strip()
dim_geography = geography_df[[
    'zip', 'city', 'region', 'district'
]].drop_duplicates(subset=['zip']).sort_values('zip').reset_index(drop=True)

print(f"✅ Dim_Geography: {dim_geography.shape[0]:,} khu vực | Khóa chính: zip")
display(dim_geography.head())

--- Đang tạo Dim_Geography ---
✅ Dim_Geography: 39,948 khu vực | Khóa chính: zip


,zip,city,region,district
0,1,Da Lat,West,District #34
1,10,Tra Vinh,West,District #34
2,100,Rach Gia,West,District #33
3,10001,Uong Bi,East,District #03
4,10002,Viet Tri,East,District #03


In [23]:
# ==============================================================================
# 3.4. TẠO DIM_PROMOTION (Khóa chính: promo_id)
# Cột gốc: promo_id, promo_name, promo_type, discount_value, start_date, end_date,
#          applicable_category, promo_channel, stackable_flag, min_order_value
# ==============================================================================
print("--- Đang tạo Dim_Promotion ---")

dim_promotion = promotions_df.drop_duplicates(subset=['promo_id']).sort_values('promo_id').reset_index(drop=True)

print(f"✅ Dim_Promotion: {dim_promotion.shape[0]:,} khuyến mãi | Khóa chính: promo_id")
display(dim_promotion.head())

--- Đang tạo Dim_Promotion ---
✅ Dim_Promotion: 50 khuyến mãi | Khóa chính: promo_id


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,All Categories,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,All Categories,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,All Categories,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,All Categories,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


In [24]:
# ==============================================================================
# 3.5. TẠO DIM_DATE (Khóa chính: order_date dạng chuỗi 'YYYY-MM-DD')
# Bảng chiều ngày sinh tự động phục vụ phân cấp thời gian cho orders
# ==============================================================================
print("--- Đang tạo Dim_Date ---")

orders_df['order_date'] = pd.to_datetime(orders_df['order_date'], errors='coerce')
min_date = orders_df['order_date'].min()
max_date = orders_df['order_date'].max()

date_range = pd.date_range(start=min_date, end=max_date, freq='D')
dim_date = pd.DataFrame({'date_raw': date_range})

# Khóa chính order_date khớp trực tiếp với cột order_date của orders
dim_date['order_date'] = dim_date['date_raw'].dt.strftime('%Y-%m-%d')
dim_date['year'] = dim_date['date_raw'].dt.year
dim_date['quarter'] = 'Q' + dim_date['date_raw'].dt.quarter.astype(str)
dim_date['month'] = dim_date['date_raw'].dt.month
dim_date['month_name'] = dim_date['date_raw'].dt.month_name()
dim_date['year_month'] = dim_date['date_raw'].dt.strftime('%Y-%m')
dim_date['day'] = dim_date['date_raw'].dt.day
dim_date['day_name'] = dim_date['date_raw'].dt.day_name()
dim_date['day_of_week'] = dim_date['date_raw'].dt.dayofweek + 1
dim_date['is_weekend'] = np.where(dim_date['day_of_week'].isin([6, 7]), 1, 0)

dim_date = dim_date[[
    'order_date', 'year', 'quarter', 'month', 'month_name', 'year_month',
    'day', 'day_name', 'day_of_week', 'is_weekend'
]]

print(f"✅ Dim_Date: {dim_date.shape[0]:,} ngày ({min_date.strftime('%Y-%m-%d')} đến {max_date.strftime('%Y-%m-%d')}) | Khóa chính: order_date")
display(dim_date.head())

--- Đang tạo Dim_Date ---
✅ Dim_Date: 3,833 ngày (2012-07-04 đến 2022-12-31) | Khóa chính: order_date


,order_date,year,quarter,month,month_name,year_month,day,day_name,day_of_week,is_weekend
0,2012-07-04,2012,Q3,7,July,2012-07,4,Wednesday,3,0
1,2012-07-05,2012,Q3,7,July,2012-07,5,Thursday,4,0
2,2012-07-06,2012,Q3,7,July,2012-07,6,Friday,5,0
3,2012-07-07,2012,Q3,7,July,2012-07,7,Saturday,6,1
4,2012-07-08,2012,Q3,7,July,2012-07,8,Sunday,7,1


---
## 4. XUẤT CÁC BẢNG DIMENSION RA FILE CSV

In [25]:
# ==============================================================================
# 5. XUẤT BẢO CÁO CHẤT LƯỢNG VÀ TẢI FILE ZIP (GIỮ NGUYÊN TÊN CỘT GỐC)
# ==============================================================================
import os
import shutil

OUTPUT_DIR = 'Gold_Dimensions'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=================================================================")
print("          BÁO CÁO KIỂM TRA CHẤT LƯỢNG 5 BẢNG DIMENSIONS          ")
print("=================================================================\n")

# Sử dụng trực tiếp các dataframe gốc không qua rename
dims_dict = {
    'Dim_Date': (dim_date, 'order_date'),
    'Dim_Geography': (dim_geography, 'zip'),
    'Dim_Customer': (dim_customer, 'customer_id'),
    'Dim_Product': (dim_product, 'product_id'),
    'Dim_Promotion': (dim_promotion, 'promo_id'),
}

for name, (df_dim, pk_col) in dims_dict.items():
    pk_nulls = df_dim[pk_col].isnull().sum()
    pk_dups = df_dim[pk_col].duplicated().sum()

    status_pk = "PASSED ✓" if (pk_nulls == 0 and pk_dups == 0) else f"FAILED ✗ ({pk_dups} dups, {pk_nulls} nulls)"

    print(f"📦 Bảng [{name}]:")
    print(f"   • Tổng số bản ghi     : {len(df_dim):,}")
    print(f"   • Số lượng thuộc tính : {len(df_dim.columns)}")
    print(f"   • Khóa chính (PK: {pk_col}): {status_pk}")

    # Xuất file CSV UTF-8 có BOM
    output_file = os.path.join(OUTPUT_DIR, f"{name}.csv")
    df_dim.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"   • Đã lưu CSV          : '{output_file}'\n")

# Nén toàn bộ thư mục kết quả
zip_filename = 'Gold_Dimensions'
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)
print(f"✓ Đã nén toàn bộ các bảng Dimension thành file: '{zip_filename}.zip'")

# Tự động tải file zip về máy nếu đang trên Colab
try:
    from google.colab import files
    print("\n-> Đang tự động tải file 'Gold_Dimensions.zip' về máy tính của bạn...")
    files.download(f"{zip_filename}.zip")
except ImportError:
    print(f"\n-> Môi trường Local: Toàn bộ file đã được lưu tại thư mục '{OUTPUT_DIR}/' và '{zip_filename}.zip'")

print("\n=================================================================")
print("🎉 HOÀN THÀNH XUẤT TOÀN BỘ CÁC BẢNG DIMENSION (ORIGINAL COLUMNS)! ")
print("=================================================================")

          BÁO CÁO KIỂM TRA CHẤT LƯỢNG 5 BẢNG DIMENSIONS          

📦 Bảng [Dim_Date]:
   • Tổng số bản ghi     : 3,833
   • Số lượng thuộc tính : 10
   • Khóa chính (PK: order_date): PASSED ✓
   • Đã lưu CSV          : 'Gold_Dimensions/Dim_Date.csv'

📦 Bảng [Dim_Geography]:
   • Tổng số bản ghi     : 39,948
   • Số lượng thuộc tính : 4
   • Khóa chính (PK: zip): PASSED ✓
   • Đã lưu CSV          : 'Gold_Dimensions/Dim_Geography.csv'

📦 Bảng [Dim_Customer]:
   • Tổng số bản ghi     : 121,930
   • Số lượng thuộc tính : 6
   • Khóa chính (PK: customer_id): PASSED ✓
   • Đã lưu CSV          : 'Gold_Dimensions/Dim_Customer.csv'

📦 Bảng [Dim_Product]:
   • Tổng số bản ghi     : 2,412
   • Số lượng thuộc tính : 8
   • Khóa chính (PK: product_id): PASSED ✓
   • Đã lưu CSV          : 'Gold_Dimensions/Dim_Product.csv'

📦 Bảng [Dim_Promotion]:
   • Tổng số bản ghi     : 50
   • Số lượng thuộc tính : 10
   • Khóa chính (PK: promo_id): PASSED ✓
   • Đã lưu CSV          : 'Gold_Dimensions/Dim_Promot

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 HOÀN THÀNH XUẤT TOÀN BỘ CÁC BẢNG DIMENSION (ORIGINAL COLUMNS)! 
